# 12-15. X5_l1 (X5) 로그 점예측 smearing 보정 (참고)

로그 스케일의 점예측 `exp(μ)`는 중앙값이라 기댓값보다 낮다. 보정하면 점예측 오차(MSE)가 얼마나 줄어드는지 참고용으로 본다. 구간과 CRPS는 바뀌지 않으므로 선정에는 영향이 없다.

| 점예측 | 식 |
|---|---|
| 중앙값(기존) | `exp(μ)` |
| smearing | `S_h x exp(μ)`, `S_h` = 과거 오차 풀(`o' + h <= o`, 중앙값 중심화)의 `exp(오차)` 평균 |
| 정규 가정 | `exp(σ_h²/2) x exp(μ)` (σ_h는 모델의 h단계 표준편차) |

- **대상**: X5_l1, 로그 스케일, AR단독·GARCH·SV, (d, D) = (0, 0)과 (1, 0). 12-11과 같이 변수에 `100 / 표준편차`를 곱해 적합한다.
- **평가**: 채점 origin 21개, MSE는 원 스케일에서 `var_std`로 정규화(로그가 선정된 변수는 이 변수 하나)

## 1. 구현 (12-3의 일반화 구현)

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')
from dataclasses import dataclass
import numpy as np
import pandas as pd
import pmdarima as pm
from scipy import stats
from arch.univariate import ARX, ConstantMean, GARCH, ZeroMean
from scipy.optimize import minimize
XI_MEAN, XI_VAR, MAX_PHI = -1.2704, np.pi ** 2 / 2, 0.97
H, ALPHA, P_MAX = 4, 0.10, 3
Z = stats.norm.ppf(1 - ALPHA / 2)


@dataclass(frozen=True)
class Spec:
    d: int = 1
    D: int = 0
    m: int = 4
    seasonal_ar: bool = False        # True면 계절 AR 차수 P도 AR단독 auto_arima가 탐색(D > 0이면 항상 탐색)

    @property
    def const(self):
        return self.d + self.D == 0


# ---------------------------------------------------------------- 차분·역차분·σ 전파
def diff_poly(sp):
    c = np.array([1.0])
    for _ in range(sp.d):
        c = np.convolve(c, [1.0, -1.0])
    for _ in range(sp.D):
        c = np.convolve(c, np.r_[1.0, np.zeros(sp.m - 1), -1.0])
    return c


def difference(y, sp):
    c, y = diff_poly(sp), np.asarray(y, float)
    k = len(c) - 1
    return y.copy() if k == 0 else np.convolve(y, c)[k:len(y)]


def undiff(w_fc, y, sp):
    c = diff_poly(sp)
    hist, out = list(map(float, y)), []
    for wf in w_fc:
        yt = wf - sum(c[i] * hist[-i] for i in range(1, len(c)))
        hist.append(yt)
        out.append(yt)
    return np.array(out)


def level_sigma(lags, ar, sp, innov_var):
    """h단계 수준 예측오차의 표준편차. 평균식 AR 다항식과 차분 다항식을 곱한 다항식으로 충격반응(psi)을 구한다."""
    ar_poly = np.zeros(max(lags) + 1 if len(lags) else 1)
    ar_poly[0] = 1.0
    for l, a in zip(lags, ar):
        ar_poly[l] -= a
    a_full = -np.convolve(ar_poly, diff_poly(sp))[1:]
    psi = np.zeros(H)
    for j in range(H):
        psi[j] = 1.0 if j == 0 else sum(a_full[k - 1] * psi[j - k] for k in range(1, min(j, len(a_full)) + 1))
    return np.sqrt(np.array([sum(psi[h - i] ** 2 * innov_var[i] for i in range(h + 1)) for h in range(H)]))


def lag_list(p, P, m):
    return sorted(set(range(1, p + 1)) | {m * j for j in range(1, P + 1)})


# ---------------------------------------------------------------- 구조 선택 (AR단독 auto_arima)
def select_structure(train, sp):
    kw = dict(d=sp.d, start_q=0, max_q=0, max_p=P_MAX, information_criterion="aicc", with_intercept=sp.const,
              suppress_warnings=True, error_action="ignore")
    if sp.D or sp.seasonal_ar:
        m = pm.auto_arima(train, seasonal=True, m=sp.m, D=sp.D, start_Q=0, max_Q=0, **kw)
        return m.order[0], m.seasonal_order[0], m
    m = pm.auto_arima(train, seasonal=False, **kw)
    return m.order[0], 0, m


def ar_forecast(m):
    fc, conf = m.predict(n_periods=H, return_conf_int=True, alpha=ALPHA)
    return np.asarray(fc), (conf[:, 1] - conf[:, 0]) / (2 * Z)


# ---------------------------------------------------------------- GARCH 동시
def garch_forecast(train, sp, lags):
    w = difference(train, sp)
    if lags:
        am = ARX(w, lags=lags, constant=sp.const, rescale=False)
    else:
        am = ConstantMean(w, rescale=False) if sp.const else ZeroMean(w, rescale=False)
    am.volatility = GARCH(p=1, q=1)
    res = am.fit(disp="off")
    f = res.forecast(horizon=H, reindex=False)
    ar = [res.params[f"y[{l}]"] for l in lags]
    fc = undiff(f.mean.values[-1], train, sp)
    sig = level_sigma(lags, ar, sp, f.residual_variance.values[-1])
    if not (np.all(np.isfinite(fc)) and np.all(np.isfinite(sig))):
        raise ValueError("non-finite forecast")
    return fc, sig


# ---------------------------------------------------------------- SV 동시 (QML + 칼만필터)
def is_ar_stable(lags, ar):
    if not len(lags):
        return True
    mx = max(lags)
    a = np.zeros(mx)
    for l, c in zip(lags, ar):
        a[l - 1] = c
    comp = np.zeros((mx, mx))
    comp[0, :] = a
    if mx > 1:
        comp[1:, :-1] = np.eye(mx - 1)
    return np.max(np.abs(np.linalg.eigvals(comp))) < 0.999


def sv_residuals(w, lags, ar, c):
    mx = max(lags) if len(lags) else 0
    return np.array([w[t] - c - sum(a * w[t - l] for a, l in zip(ar, lags)) for t in range(mx, len(w))])


def sv_qml_negloglik(z_centered, phi, sigma_eta2):
    P0 = sigma_eta2 / (1 - phi ** 2) if abs(phi) < 0.999 else sigma_eta2 * 100
    h_pred, P_pred, loglik = 0.0, P0, 0.0
    for z in z_centered:
        F = P_pred + XI_VAR
        v = z - h_pred
        loglik += -0.5 * (np.log(2 * np.pi * F) + v ** 2 / F)
        K = P_pred / F
        h_filt, P_filt = h_pred + K * v, P_pred * (1 - K)
        h_pred, P_pred = phi * h_filt, phi ** 2 * P_filt + sigma_eta2
    return -loglik


def _unpack(params, k, const):
    ar = params[:k]
    c = params[k] if const else 0.0
    j = k + (1 if const else 0)
    return ar, c, MAX_PHI * np.tanh(params[j]), np.exp(params[j + 1]) ** 2


def joint_sv_negloglik(params, w, lags, const):
    ar, c, phi, s2 = _unpack(params, len(lags), const)
    if not is_ar_stable(lags, ar):
        return 1e10
    z = np.log(sv_residuals(w, lags, ar, c) ** 2 + 1e-8)
    return sv_qml_negloglik(z - z.mean(), phi, s2)


def sv_forecast(train, sp, lags):
    w = difference(train, sp)
    k = len(lags)
    x0 = [0.0] * k + ([float(np.mean(w))] if sp.const else []) + [1.5, np.log(0.3)]
    res = minimize(joint_sv_negloglik, x0, args=(w, lags, sp.const), method="Nelder-Mead", options={"maxiter": 3000})
    ar, c, phi, s2 = _unpack(res.x, k, sp.const)
    if not is_ar_stable(lags, ar) or res.fun >= 1e9:
        raise ValueError("SV fit failed")
    z = np.log(sv_residuals(w, lags, ar, c) ** 2 + 1e-8)
    zm = z.mean()
    P, h = s2 / (1 - phi ** 2), 0.0
    for x in z - zm:
        K = P / (P + XI_VAR)
        hf, Pf = h + K * (x - h), P * (1 - K)
        h, P = phi * hf, phi ** 2 * Pf + s2
    past = list(w[-max(lags):]) if k else []
    mean_w = []
    for _ in range(H):
        pred = c + sum(a * past[-l] for a, l in zip(ar, lags))
        mean_w.append(pred)
        past.append(pred)
    v = [np.exp(zm - XI_MEAN + phi ** j * hf + (phi ** (2 * j) * Pf + s2 * sum(phi ** (2 * i) for i in range(j))) / 2) for j in range(1, H + 1)]
    fc, sig = undiff(mean_w, train, sp), level_sigma(lags, ar, sp, v)
    if not (np.all(np.isfinite(fc)) and np.all(np.isfinite(sig))):
        raise ValueError("non-finite forecast")
    return fc, sig


# ---------------------------------------------------------------- 공용 진입점
def forecast(model, train, sp=Spec()):
    """model: 'AR단독' | 'GARCH' | 'SV'. 반환: (예측 4개, σ 4개, 사용한 모델, (p, P)). 적합 실패 시 NaN을 돌려준다(src='실패')."""
    p, P, m = select_structure(train, sp)
    if model == "AR단독":
        fc, sig = ar_forecast(m)
        return fc, sig, "AR단독", (p, P)
    lags = lag_list(p, P, sp.m)
    try:
        fc, sig = (garch_forecast if model == "GARCH" else sv_forecast)(train, sp, lags)
        return fc, sig, model, (p, P)
    except Exception:
        nan = np.full(H, np.nan)
        return nan, nan, "실패", (p, P)


## 2. 계산

In [2]:
raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)
y0 = raw[["quarter", "X5_l1"]].dropna()["X5_l1"].values.astype(float)
vals = y0 * (100 / y0.std())
SCALE = 100.0
LOG = SCALE * np.log(vals)
var_std = vals.std()
origins = list(range(20, len(vals) - H + 1))
scored = origins[15:]
CASES = [(0, 0), (1, 0)]
MODELS = ["AR단독", "GARCH", "SV"]


def run(model, sp):
    errs = {h: [] for h in range(1, H + 1)}                    # (o', 로그 단위 오차)
    rows = []
    for o in origins:
        fc, sig, _, _ = forecast(model, LOG[:o], sp)
        a_log = LOG[o:o + H]
        if o in scored:
            for h in range(1, H + 1):
                pool = np.array([e for o2, e in errs[h] if o2 + h <= o])
                S = np.mean(np.exp((pool - np.median(pool)) / SCALE))
                med = np.exp(fc[h - 1] / SCALE)
                rows.append({"모델": model, "d": sp.d, "D": sp.D, "origin": o, "h": h, "S": S, "실제": vals[o + h - 1],
                             "중앙값": med, "smearing": S * med, "정규 가정": med * np.exp((sig[h - 1] / SCALE) ** 2 / 2)})
        for h in range(1, H + 1):
            errs[h].append((o, a_log[h - 1] - fc[h - 1]))
    return pd.DataFrame(rows)


CACHE = "../cache/cache_1214_smearing_df.pkl"
if os.path.exists(CACHE):
    res = pd.read_pickle(CACHE)
else:
    res = pd.concat([run(m, Spec(d=d, D=D)) for m in MODELS for d, D in CASES], ignore_index=True)
    res.to_pickle(CACHE)
print(len(res), "행")

504 행


## 3. 점예측 MSE 비교

원 스케일 열은 같은 모델과 (d, D)를 원 스케일에서 적합한 MSE(12-11, 참고)다. `평균 오차`는 (실제 - 점예측) / var_std의 평균으로, 음수면 점예측이 실제보다 높은 쪽이다.

In [3]:
for k in ("중앙값", "smearing", "정규 가정"):
    res[f"se_{k}"] = ((res["실제"] - res[k]) / var_std) ** 2
    res[f"e_{k}"] = (res["실제"] - res[k]) / var_std
r_ = pd.read_pickle("../cache/cache_1210_X5_l1_df.pkl")
r_ = r_[(r_["스케일"] == "원") & (r_["구간"] == "자체")]
orig = {(m, d, D): g["MSE"].mean() for (m, d, D), g in r_.groupby(["모델", "d", "D"])}
rows = []
for (m, d, D), g in res.groupby(["모델", "d", "D"]):
    rows.append({"모델": m, "d": d, "D": D, "MSE 중앙값": g["se_중앙값"].mean(), "MSE smearing": g["se_smearing"].mean(), "MSE 정규 가정": g["se_정규 가정"].mean(),
                 "MSE 원 스케일(참고)": orig[(m, d, D)], "smearing 개선(%)": 100 * (g["se_smearing"].mean() / g["se_중앙값"].mean() - 1),
                 "평균 오차 중앙값": g["e_중앙값"].mean(), "평균 오차 smearing": g["e_smearing"].mean(), "평균 S": g["S"].mean()})
tab = pd.DataFrame(rows).set_index(["모델", "d", "D"]).sort_index(level=[1, 0], ascending=[True, True])
with pd.option_context("display.float_format", "{:,.4f}".format, "display.width", 250):
    display(tab)

,,,MSE 중앙값,MSE smearing,MSE 정규 가정,MSE 원 스케일(참고),smearing 개선(%),평균 오차 중앙값,평균 오차 smearing,평균 S
모델,d,D,,,,,,,,
AR단독,0,0,1.9304,2.3907,1.9821,2.1905,23.8425,0.2636,-0.2397,1.2801
GARCH,0,0,1.8950,2.1075,1.9804,2.1114,11.2092,0.3641,-0.1200,1.2899
SV,0,0,2.5308,4.0068,17.1603,3.2012,58.3208,-0.1993,-0.8522,1.2955
AR단독,1,0,2.8551,5.2848,3.8670,2.8551,85.1004,-0.0546,-0.7871,1.3449
GARCH,1,0,2.8551,5.2848,4.8758,2.8551,85.1004,-0.0546,-0.7871,1.3449
SV,1,0,2.8551,5.2848,3.2178,2.8551,85.1004,-0.0546,-0.7871,1.3449


## 4. h별 보정 계수와 개선

`S`는 h마다 다르다. 개선이 어느 예측 단계에서 나는지, origin별로 일관된지 본다.

In [4]:
g = res[(res["d"] == 0) & (res["D"] == 0)]
by_h = g.groupby(["모델", "h"]).agg(S=("S", "mean"), 중앙값=("se_중앙값", "mean"), smearing=("se_smearing", "mean"))
by_h["개선(%)"] = 100 * (by_h["smearing"] / by_h["중앙값"] - 1)
with pd.option_context("display.float_format", "{:,.4f}".format):
    display(by_h)
po = res.groupby(["모델", "d", "D", "origin"])[["se_중앙값", "se_smearing"]].mean()
w = (po["se_smearing"] < po["se_중앙값"]).groupby(level=[0, 1, 2]).agg(["sum", "count"])
w.columns = ["smearing이 더 나은 origin", "origin 수"]
display(w)

S    중앙값  smearing   개선(%)
모델    h                                
AR단독  1 1.1107 1.1732    1.2069  2.8703
      2 1.1776 2.0146    2.1557  7.0023
      3 1.2826 2.4620    2.6172  6.3066
      4 1.5496 2.0718    3.5828 72.9320
GARCH 1 1.1057 1.1538    1.1510 -0.2453
      2 1.1917 1.9613    1.9989  1.9177
      3 1.3014 2.4142    2.4307  0.6857
      4 1.5607 2.0509    2.8492 38.9261
SV    1 1.4364 2.7996    4.5799 63.5894
      2 1.2237 2.2766    3.3155 45.6349
      3 1.2834 2.6781    4.3316 61.7408
      4 1.2383 2.3688    3.8001 60.4195

smearing이 더 나은 origin  origin 수
모델    d D                                 
AR단독  0 0                      6        21
      1 0                      7        21
GARCH 0 0                      8        21
      1 0                      7        21
SV    0 0                      4        21
      1 0                      7        21

## 결론

- **선정된 구조 (d, D) = (0, 0)에서도 smearing은 MSE를 나쁘게 한다**: AR단독 1.930 -> 2.391(+24%), GARCH 1.895 -> 2.108(+11%), SV 2.531 -> 4.007(+58%). 선정 모델인 GARCH는 +11%이고 origin 21개 중 8개에서만 더 낫다.
- **보정이 과하다**: 중앙값 점예측의 평균 오차는 +0.26~+0.36(실제보다 낮은 쪽)이지만 `S`가 1.28~1.30이라 보정 후 -0.12~-0.24로 반대쪽으로 넘어간다. 편향은 있어도 `S`가 그 크기보다 크다.
- **h=4에서 악화가 가장 크다**: `S`가 h=4에서 약 1.55로 가장 크고 AR단독 +73%, GARCH +39%다. h=1~3은 거의 같다.
- **d=1은 세 모델의 점예측이 같다**: 세 모델 모두 AR 차수가 0이라 점예측이 같고(랜덤워크), MSE는 2.855 -> 5.285(+85%)로 악화한다.
- **판단**: X3_l1(12-14)과 같이 도입하지 않는다. 로그가 선정된 변수 두 곳 모두에서 선정된 구조의 중앙값 점예측이 보정 점예측보다 낫다. 점예측을 기댓값으로 써야 하면 별도로 검토한다.
